# 实验三：评论识别问题 - 概率模型

## 实验题目

给定 8 条历史评论数据，从评论中提取三个关键词特征：是否包含“好”、是否包含“快”、是否包含“坏”。情感标签分为“好评”和“差评”。要求手动建立朴素贝叶斯分类模型，并判断新评论 $X=\{包含好=1, 包含快=1, 包含坏=0\}$ 的情感类别。


## 问题背景

评论识别是概率分类模型的典型应用。对于一条新评论，如果它包含“好”“快”等正向词，且不包含“坏”等负向词，通常更可能是好评。朴素贝叶斯模型可以把这种直觉转化为概率计算：先统计历史数据中各类评论出现的比例，再统计不同类别下关键词出现的概率，最后用贝叶斯公式计算新评论属于各类别的相对可能性。


## 数学模型建立

设情感类别为 $C\in\{好评, 差评\}$，评论特征向量为：

$$X=(x_1,x_2,x_3)$$

其中：

- $x_1$：是否包含“好”
- $x_2$：是否包含“快”
- $x_3$：是否包含“坏”

贝叶斯公式为：

$$P(C|X)=\frac{P(C)P(X|C)}{P(X)}$$

对分类而言，$P(X)$ 对所有类别相同，因此只需比较：

$$P(C|X)\propto P(C)P(X|C)$$

朴素贝叶斯模型假设在给定类别后，三个关键词特征相互独立，因此：

$$P(X|C)=P(x_1|C)P(x_2|C)P(x_3|C)$$

本实验使用拉普拉斯平滑避免概率为 0。对于二值特征，平滑后的条件概率为：

$$P(x_j=1|C)=\frac{count(x_j=1,C)+\alpha}{count(C)+2\alpha}$$

$$P(x_j=0|C)=\frac{count(x_j=0,C)+\alpha}{count(C)+2\alpha}$$

这里取 $\alpha=1$。


## 算法设计思路

1. 将历史评论数据写入 `pandas DataFrame`。
2. 将“是/否”转换为 1/0。
3. 计算类别先验概率 $P(好评)$ 和 $P(差评)$。
4. 对每个类别、每个特征计算拉普拉斯平滑后的条件概率。
5. 对新评论分别计算“好评”和“差评”的未归一化后验概率。
6. 对两个概率归一化，并选择概率更大的类别作为预测结果。


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

OUTPUT_DIR = Path.cwd().parent / 'outputs' if Path.cwd().name == 'notebooks' else Path('math_modeling_experiments/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Python 程序实现

### 1. 将历史数据写入 DataFrame


In [ ]:
raw_data = [
    ['评论1', '质量很好，物流快', '是', '是', '否', '好评'],
    ['评论2', '东西好，但是有点贵', '是', '否', '否', '好评'],
    ['评论3', '物流非常快，满意', '否', '是', '否', '好评'],
    ['评论4', '非常好的一次购物', '是', '否', '否', '好评'],
    ['评论5', '东西坏了，很差劲', '否', '否', '是', '差评'],
    ['评论6', '物流太慢，东西坏', '否', '否', '是', '差评'],
    ['评论7', '包装坏了，体验不好', '否', '否', '是', '差评'],
    ['评论8', '速度慢，服务坏', '否', '否', '是', '差评'],
]

columns = ['编号', '评论内容', '包含好', '包含快', '包含坏', '情感标签']
df_raw = pd.DataFrame(raw_data, columns=columns)
df_raw


### 2. 将“是/否”转换为 1/0

三个关键词特征都是二值变量。为了便于计算，将“是”记为 1，“否”记为 0。


In [ ]:
feature_cols = ['包含好', '包含快', '包含坏']
df = df_raw.copy()
df[feature_cols] = df[feature_cols].apply(lambda col: col.map({'是': 1, '否': 0})).astype(int)
df


### 3. 计算类别先验概率

先验概率表示在没有观察到新评论特征之前，一条评论属于某个类别的概率。


In [ ]:
class_labels = ['好评', '差评']
alpha = 1

prior_probs = df['情感标签'].value_counts(normalize=True).reindex(class_labels)
prior_table = prior_probs.reset_index()
prior_table.columns = ['情感类别', '先验概率']
prior_table


### 4. 计算条件概率表

对每个类别分别统计三个关键词的出现情况，并使用 $\alpha=1$ 的拉普拉斯平滑。


In [ ]:
def compute_conditional_probabilities(data, features, labels, alpha=1):
    records = []
    probability_lookup = {}
    for label in labels:
        subset = data[data['情感标签'] == label]
        class_count = len(subset)
        probability_lookup[label] = {}
        for feature in features:
            count_1 = int(subset[feature].sum())
            count_0 = class_count - count_1
            prob_1 = (count_1 + alpha) / (class_count + 2 * alpha)
            prob_0 = (count_0 + alpha) / (class_count + 2 * alpha)
            probability_lookup[label][(feature, 1)] = prob_1
            probability_lookup[label][(feature, 0)] = prob_0
            records.append({
                '情感类别': label,
                '特征': feature,
                '取值': 1,
                '计数': count_1,
                '平滑后条件概率': prob_1,
            })
            records.append({
                '情感类别': label,
                '特征': feature,
                '取值': 0,
                '计数': count_0,
                '平滑后条件概率': prob_0,
            })
    return pd.DataFrame(records), probability_lookup


conditional_table, conditional_probs = compute_conditional_probabilities(
    df, feature_cols, class_labels, alpha=alpha
)
conditional_table


为了更清楚地查看模型参数，下面把条件概率整理成宽表形式。


In [ ]:
conditional_wide = conditional_table.copy()
conditional_wide['条件'] = conditional_wide['特征'] + '=' + conditional_wide['取值'].astype(str)
conditional_wide = conditional_wide.pivot(index='条件', columns='情感类别', values='平滑后条件概率')
conditional_wide = conditional_wide[class_labels]
conditional_wide


## 新评论分类计算

设新评论 $X=\{包含好=1, 包含快=1, 包含坏=0\}$。根据朴素贝叶斯规则：

$$P(好评|X)\propto P(好评)P(包含好=1|好评)P(包含快=1|好评)P(包含坏=0|好评)$$

$$P(差评|X)\propto P(差评)P(包含好=1|差评)P(包含快=1|差评)P(包含坏=0|差评)$$


In [ ]:
def predict_comment(features, prior_probs=prior_probs, conditional_probs=conditional_probs, labels=class_labels):
    """输入三个布尔或 0/1 特征，返回朴素贝叶斯预测结果。"""
    feature_values = {feature: int(value) for feature, value in features.items()}
    rows = []
    unnormalized = {}
    for label in labels:
        prob = float(prior_probs[label])
        terms = [f'P({label})={prob:.6f}']
        for feature in feature_cols:
            value = feature_values[feature]
            cond_prob = conditional_probs[label][(feature, value)]
            prob *= cond_prob
            terms.append(f'P({feature}={value}|{label})={cond_prob:.6f}')
        unnormalized[label] = prob
        rows.append({
            '情感类别': label,
            '计算过程': ' × '.join(terms),
            '未归一化概率': prob,
        })

    total = sum(unnormalized.values())
    for row in rows:
        row['归一化后验概率'] = row['未归一化概率'] / total

    result_table = pd.DataFrame(rows)
    prediction = result_table.sort_values('归一化后验概率', ascending=False).iloc[0]['情感类别']
    return prediction, result_table


new_comment_features = {'包含好': 1, '包含快': 1, '包含坏': 0}
prediction, posterior_table = predict_comment(new_comment_features)
posterior_table


## 结果输出

下面输出新评论的后验概率和最终分类结果，并进行参考结果校验。


In [ ]:
positive_probability = posterior_table.loc[
    posterior_table['情感类别'] == '好评', '归一化后验概率'
].iloc[0]
negative_probability = posterior_table.loc[
    posterior_table['情感类别'] == '差评', '归一化后验概率'
].iloc[0]

print('新评论特征：', new_comment_features)
print(f'P(好评 | X) = {positive_probability:.3f}')
print(f'P(差评 | X) = {negative_probability:.3f}')
print('最终分类结果：', prediction)

assert prior_probs['好评'] == 0.5
assert prior_probs['差评'] == 0.5
assert prediction == '好评'
assert abs(positive_probability - 0.984) < 0.002


## 表格展示概率计算过程

下表汇总先验概率、条件概率和后验概率，便于检查每一步计算。


In [ ]:
summary_rows = []
for label in class_labels:
    row = {'情感类别': label, '先验概率': prior_probs[label]}
    for feature, value in new_comment_features.items():
        row[f'P({feature}={value}|{label})'] = conditional_probs[label][(feature, value)]
    posterior_row = posterior_table[posterior_table['情感类别'] == label].iloc[0]
    row['未归一化概率'] = posterior_row['未归一化概率']
    row['归一化后验概率'] = posterior_row['归一化后验概率']
    summary_rows.append(row)

calculation_summary = pd.DataFrame(summary_rows)
calculation_summary


## 图表展示

下面用柱状图展示新评论属于“好评”和“差评”的归一化后验概率，并保存到 `outputs/` 文件夹。


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
plot_data = posterior_table.set_index('情感类别')['归一化后验概率']
colors = ['#2a9d8f' if label == prediction else '#e76f51' for label in plot_data.index]
plot_data.plot(kind='bar', color=colors, ax=ax)
ax.set_ylim(0, 1.05)
ax.set_ylabel('归一化后验概率')
ax.set_title('新评论情感分类后验概率')
ax.grid(axis='y', alpha=0.25)
plt.xticks(rotation=0)
for i, value in enumerate(plot_data.values):
    ax.text(i, value + 0.025, f'{value:.3f}', ha='center')
plt.tight_layout()
fig_path = OUTPUT_DIR / '03_bayes_comment_posterior.png'
plt.savefig(fig_path, dpi=150)
plt.show()
print('图表已保存：', fig_path)


## 可选：输入任意三个布尔特征进行预测

下面给出几个示例，展示 `predict_comment(features)` 函数可以对不同关键词组合进行分类。


In [ ]:
examples = [
    {'包含好': 1, '包含快': 1, '包含坏': 0},
    {'包含好': 0, '包含快': 0, '包含坏': 1},
    {'包含好': 1, '包含快': 0, '包含坏': 1},
    {'包含好': 0, '包含快': 1, '包含坏': 0},
]

example_results = []
for features in examples:
    pred, table = predict_comment(features)
    example_results.append({
        '包含好': features['包含好'],
        '包含快': features['包含快'],
        '包含坏': features['包含坏'],
        'P(好评|X)': table.loc[table['情感类别'] == '好评', '归一化后验概率'].iloc[0],
        'P(差评|X)': table.loc[table['情感类别'] == '差评', '归一化后验概率'].iloc[0],
        '预测结果': pred,
    })

pd.DataFrame(example_results)


## 结论分析

使用 $\alpha=1$ 的拉普拉斯平滑后，历史数据中 $P(好评)=0.5$，$P(差评)=0.5$。对于新评论 $X=\{包含好=1, 包含快=1, 包含坏=0\}$，模型计算得到好评的归一化后验概率约为 0.984，明显大于差评概率，因此应判断为“好评”。这说明在本题给定的小样本和三个关键词特征下，朴素贝叶斯模型能够有效利用“好”“快”“坏”三个关键词进行情感识别。

不过，该模型也存在明显局限。第一，样本量只有 8 条，统计概率不够稳定；第二，特征过少，不能表达更多情感信息；第三，简单关键词规则容易误判，例如“体验不好”中包含“好”，但整体语义是负向；第四，朴素独立性假设忽略了词语之间的组合关系。后续可以加入中文分词、更多关键词、否定词处理、TF-IDF 特征，或使用更复杂的机器学习文本分类模型提升效果。


In [ ]:
# 整合整个实验的关键信息和结果表格
integrated_rows = []

# 1. 数据集与特征概况
integrated_rows.append({
    '信息模块': '数据集概况',
    '结果来源': 'df_raw / df',
    '指标': '样本总数',
    '情感类别': '全部',
    '特征或条件': '-',
    '取值': '-',
    '计数': len(df_raw),
    '概率或结果': len(df_raw),
    '说明': '历史评论样本数量',
})

for label in class_labels:
    class_count = int((df['情感标签'] == label).sum())
    integrated_rows.append({
        '信息模块': '数据集概况',
        '结果来源': 'df',
        '指标': '类别样本数',
        '情感类别': label,
        '特征或条件': '情感标签',
        '取值': label,
        '计数': class_count,
        '概率或结果': class_count,
        '说明': f'{label} 样本数量',
    })

for feature in feature_cols:
    feature_count = int(df[feature].sum())
    integrated_rows.append({
        '信息模块': '数据集概况',
        '结果来源': 'df',
        '指标': '关键词出现次数',
        '情感类别': '全部',
        '特征或条件': feature,
        '取值': 1,
        '计数': feature_count,
        '概率或结果': feature_count / len(df),
        '说明': f'包含关键词“{feature.replace("包含", "")}”的样本占比',
    })

# 2. 类别先验概率
for _, row in prior_table.iterrows():
    integrated_rows.append({
        '信息模块': '先验概率',
        '结果来源': 'prior_table',
        '指标': 'P(情感类别)',
        '情感类别': row['情感类别'],
        '特征或条件': '情感标签',
        '取值': row['情感类别'],
        '计数': int((df['情感标签'] == row['情感类别']).sum()),
        '概率或结果': row['先验概率'],
        '说明': '不观察新评论特征时的类别概率',
    })

# 3. 拉普拉斯平滑后的条件概率
for _, row in conditional_table.iterrows():
    integrated_rows.append({
        '信息模块': '条件概率',
        '结果来源': 'conditional_table',
        '指标': f"P({row['特征']}={row['取值']}|{row['情感类别']})",
        '情感类别': row['情感类别'],
        '特征或条件': row['特征'],
        '取值': int(row['取值']),
        '计数': int(row['计数']),
        '概率或结果': row['平滑后条件概率'],
        '说明': f'alpha={alpha} 的拉普拉斯平滑结果',
    })

# 4. 新评论的后验概率和最终分类
new_feature_text = '，'.join([f'{feature}={value}' for feature, value in new_comment_features.items()])
for _, row in posterior_table.iterrows():
    integrated_rows.append({
        '信息模块': '新评论预测',
        '结果来源': 'posterior_table',
        '指标': f"P({row['情感类别']}|X)",
        '情感类别': row['情感类别'],
        '特征或条件': new_feature_text,
        '取值': 'X',
        '计数': '-',
        '概率或结果': row['归一化后验概率'],
        '说明': f"未归一化概率={row['未归一化概率']:.6f}",
    })

integrated_rows.append({
    '信息模块': '新评论预测',
    '结果来源': 'prediction',
    '指标': '最终分类结果',
    '情感类别': prediction,
    '特征或条件': new_feature_text,
    '取值': 'X',
    '计数': '-',
    '概率或结果': prediction,
    '说明': '选择归一化后验概率最大的情感类别',
})

# 5. 概率计算过程宽表中的核心结果
for _, row in calculation_summary.iterrows():
    integrated_rows.append({
        '信息模块': '计算过程汇总',
        '结果来源': 'calculation_summary',
        '指标': '新评论分类计算汇总',
        '情感类别': row['情感类别'],
        '特征或条件': new_feature_text,
        '取值': 'X',
        '计数': '-',
        '概率或结果': row['归一化后验概率'],
        '说明': f"先验={row['先验概率']:.6f}，未归一化概率={row['未归一化概率']:.6f}",
    })

# 6. 多组示例输入的预测结果
example_prediction_df = pd.DataFrame(example_results)
for idx, row in example_prediction_df.iterrows():
    feature_text = f"包含好={int(row['包含好'])}，包含快={int(row['包含快'])}，包含坏={int(row['包含坏'])}"
    integrated_rows.append({
        '信息模块': '示例预测',
        '结果来源': 'example_prediction_df',
        '指标': f'示例 {idx + 1} 预测结果',
        '情感类别': row['预测结果'],
        '特征或条件': feature_text,
        '取值': 'X',
        '计数': '-',
        '概率或结果': row['预测结果'],
        '说明': f"P(好评|X)={row['P(好评|X)']:.6f}，P(差评|X)={row['P(差评|X)']:.6f}",
    })

key_information_summary_df = pd.DataFrame(integrated_rows)
key_information_summary_display = key_information_summary_df.copy()
key_information_summary_display['概率或结果'] = key_information_summary_display['概率或结果'].apply(
    lambda x: f'{x:.6f}' if isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, bool) else x
)

display(key_information_summary_display)


In [ ]:
# 一行两图整合展示：关键词条件概率与测试评论后验概率
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

class_order = ['好评', '差评']
keyword_info = [('包含好', '好'), ('包含快', '快'), ('包含坏', '坏')]

condition_fallback = {
    ('包含好', '好评'): 0.666667,
    ('包含好', '差评'): 0.166667,
    ('包含快', '好评'): 0.500000,
    ('包含快', '差评'): 0.166667,
    ('包含坏', '好评'): 0.166667,
    ('包含坏', '差评'): 0.833333,
}
posterior_fallback = {'好评': 0.983607, '差评': 0.016393}


def get_condition_probability(feature, label):
    """优先从 conditional_table 提取 P(feature=1|label)，若变量缺失则使用给定关键数据。"""
    if 'conditional_table' in globals():
        mask = (
            (conditional_table['特征'] == feature)
            & (conditional_table['情感类别'] == label)
            & (conditional_table['取值'].astype(int) == 1)
        )
        matched = conditional_table.loc[mask, '平滑后条件概率']
        if not matched.empty:
            return float(matched.iloc[0])
    return condition_fallback[(feature, label)]


def get_posterior_probability(label):
    """优先从 posterior_table 提取 P(label|X)，若变量缺失则使用给定关键数据。"""
    if 'posterior_table' in globals():
        matched = posterior_table.loc[posterior_table['情感类别'] == label, '归一化后验概率']
        if not matched.empty:
            return float(matched.iloc[0])
    return posterior_fallback[label]

condition_probs = {
    label: [get_condition_probability(feature, label) for feature, _ in keyword_info]
    for label in class_order
}
posterior_probs = {label: get_posterior_probability(label) for label in class_order}
prediction_label = prediction if 'prediction' in globals() else max(posterior_probs, key=posterior_probs.get)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 左图：关键词条件概率对比
ax = axes[0]
x = np.arange(len(keyword_info))
width = 0.36
bars_positive = ax.bar(x - width / 2, condition_probs['好评'], width, label='好评', color='#2a9d8f')
bars_negative = ax.bar(x + width / 2, condition_probs['差评'], width, label='差评', color='#e76f51')

ax.set_title('关键词条件概率对比')
ax.set_xlabel('关键词')
ax.set_ylabel('条件概率')
ax.set_xticks(x)
ax.set_xticklabels([keyword for _, keyword in keyword_info])
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.25)
ax.legend(title='情感类别')

for bars in (bars_positive, bars_negative):
    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + 0.025,
            f'{height:.4f}',
            ha='center',
            va='bottom',
            fontsize=9,
        )

# 右图：测试评论后验概率对比
ax = axes[1]
posterior_values = [posterior_probs[label] for label in class_order]
posterior_colors = ['#2a9d8f' if label == prediction_label else '#e76f51' for label in class_order]
bars = ax.bar(class_order, posterior_values, color=posterior_colors, width=0.5)

ax.set_title('测试评论后验概率对比')
ax.set_xlabel('情感类别')
ax.set_ylabel('后验概率')
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.25)
ax.text(
    0.5,
    0.91,
    f'预测结果：{prediction_label}',
    transform=ax.transAxes,
    ha='center',
    va='center',
    fontsize=11,
    fontweight='bold',
    color='#264653',
)

for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height + 0.025,
        f'{height:.4f}',
        ha='center',
        va='bottom',
        fontsize=9,
    )

fig.text(
    0.5,
    0.02,
    "结论：‘好’和‘快’主要支持好评判断，‘坏’主要支持差评判断；测试评论 X 的好评后验概率为 0.9836，因此判定为好评。",
    ha='center',
    fontsize=10,
)

plt.tight_layout(rect=[0, 0.10, 1, 1])
visual_summary_path = (OUTPUT_DIR if 'OUTPUT_DIR' in globals() else Path('outputs')) / 'comment_bayes_visual_summary.png'
visual_summary_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(visual_summary_path, dpi=300, bbox_inches='tight')
plt.show()
